# Jalur Penelitian R05: Efficient Computer Vision & Edge AI Systems
## Reproducible Research Baseline: Affine Quantization, Roofline Bound Analysis & Edge Latency Profiling

**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Level:** L4 Research-Ready / ISVWS 2026–2040 Standard  
**Evidence Category:** Educational Simulation & Reproducible Benchmark Template

Notebook ini menyediakan protokol komputasi deterministik untuk mereproduksi eksperimen efisiensi edge AI:
1. **Kuantisasi Affin Asimetris (FP32 -> INT8/INT4):** Perhitungan skala kuantisasi $S$ dan zero-point $Z$ serta dekuantisasi.
2. **Analisis Model Roofline:** Karakterisasi throughput komputasi terhadap arithmetic intensity ($FLOPs/Byte$) dan bandwidth memori.
3. **Protokol Profil Latensi Multi-Seed ($N \ge 3$):** Pelaporan distribusi latensi P50, P95, P99 dan faktor thermal throttling.
4. **Analisis Pareto Frontier:** Trade-off akurasi mAP terhadap kebutuhan energi per inferensi ($mJ/frame$).

In [ ]:
import os, random, numpy as np, math

def set_deterministic_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"[REPRODUCIBILITY] Random Seed locked to {seed} successfully.")

set_deterministic_seed(42)

In [ ]:
# 1. Algoritma Kuantisasi Affin Asimetris (FP32 -> INT8)
def quantize_affine(weights_fp32: np.ndarray, bits: int = 8):
    qmin = 0
    qmax = (1 << bits) - 1
    rmin = float(np.min(weights_fp32))
    rmax = float(np.max(weights_fp32))
    scale = (rmax - rmin) / float(qmax - qmin)
    if scale == 0:
        scale = 1e-8
    zero_point = int(np.round(-rmin / scale))
    zero_point = max(qmin, min(qmax, zero_point))
    
    q_int = np.clip(np.round(weights_fp32 / scale) + zero_point, qmin, qmax).astype(np.int32)
    dequant_fp32 = scale * (q_int - zero_point)
    mse_loss = float(np.mean((weights_fp32 - dequant_fp32) ** 2))
    return q_int, dequant_fp32, scale, zero_point, mse_loss

# Uji coba kuantisasi tensor bobot sintetik
synthetic_weights = np.random.uniform(-1.0, 1.0, size=(100, 100))
q_int8, dequant, s, zp, mse = quantize_affine(synthetic_weights, bits=8)
print(f"INT8 Scale: {s:.6f}, Zero-Point: {zp}, Requantization MSE: {mse:.6e}")

In [ ]:
# 2. Model Batas Komputasi Roofline
class EdgeRooflineModel:
    def __init__(self, peak_gflops: float = 100.0, mem_bw_gbps: float = 25.6):
        self.peak_gflops = peak_gflops
        self.mem_bw_gbps = mem_bw_gbps
        self.ridge_point = peak_gflops / mem_bw_gbps

    def evaluate_layer(self, flops: float, bytes_transferred: float):
        ai = flops / bytes_transferred
        achievable_perf = min(self.peak_gflops, self.mem_bw_gbps * ai)
        regime = "Compute-Bound" if ai >= self.ridge_point else "Memory-Bound"
        return ai, achievable_perf, regime

roofline = EdgeRooflineModel(peak_gflops=100.0, mem_bw_gbps=25.6)
print(f"Roofline Ridge Point (Flops/Byte): {roofline.ridge_point:.2f}")
ai_conv, perf_conv, reg_conv = roofline.evaluate_layer(flops=5e7, bytes_transferred=5e6)
print(f"Dense Conv Layer: AI={ai_conv:.1f} -> {reg_conv} @ {perf_conv:.1f} GFLOPS")
ai_dw, perf_dw, reg_dw = roofline.evaluate_layer(flops=1e6, bytes_transferred=2e6)
print(f"Depthwise Layer:  AI={ai_dw:.2f} -> {reg_dw} @ {perf_dw:.1f} GFLOPS")

In [ ]:
# 3. Profil Latensi Multi-Seed Deterministik (P50, P95, P99)
seeds = [42, 1337, 2026]
print("=== REPRODUCIBILITY BENCHMARK MULTI-SEED (GATE 6) ===")
for seed in seeds:
    set_deterministic_seed(seed)
    latencies_fp32 = np.random.normal(32.0, 5.0, 500)
    latencies_int8 = np.random.normal(18.0, 3.0, 500)
    p95_fp32 = np.percentile(latencies_fp32, 95)
    p95_int8 = np.percentile(latencies_int8, 95)
    speedup = p95_fp32 / p95_int8
    print(f"Seed {seed} -> P95 FP32: {p95_fp32:.1f}ms | INT8: {p95_int8:.1f}ms | Speedup: {speedup:.2f}x")